# PFC · A v3 no lote 2 com outros modelos (Colab)

Rodada de **teste** da v3, no **lote 2** (945 consultas novas, construídas como o lote 1, com outra semente, outros redatores e outros anotadores; nenhuma delas foi usada no desenvolvimento da v3). Extensão do teste a outros modelos, com a MESMA v3 congelada: Gemma 4 E2B e Qwen 3 4B, na T4, com a comparação principal (TC v3 × SE v3) e o A/B da v1 (TC v1 × SE v1). Plano em `docs/v3.md`, seção 'Extensão a outros modelos'. Metodologia em `docs/v3.md`.

Os degraus do Tool Calling, cada um acrescentando uma peça ao anterior:

| Degrau | Configuração | O que acrescenta |
|---|---|---|
| 0 | TC v1 | a solução do Cap. 4 |
| 1 | TC v2 | descrições com as convenções do manual + ferramenta de recusa |
| 2 | TC v3 descrições | descrições corrigidas com os erros do lote 1 |
| 3 | TC v3 + ferramentas auxiliares | identificar_nome, normalizar_codigo, normalizar_escala, resolver_periodo e pedir_esclarecimento (usuário simulado) |
| 4 | **TC v3** | + retorno: a busca confere os parâmetros e devolve erros e avisos; a recusa de consulta com critério do catálogo volta uma vez |
| controle | **SE v3** | as mesmas descrições e o mesmo retorno, como nova tentativa, sem Tool Calling |
| referências | SE v1, SE v2 | já medidas no lote 1; TC v2 + SE v1 formam a arquitetura em duas etapas simulada |

**Pré-registro:** a célula 3 confere que a v3 do pacote é exatamente a congelada antes do teste (SHA-256 abaixo). Leva cerca de 2 h 30 min; deixe a aba aberta. Se a sessão cair, `Executar tudo` de novo: o avaliador retoma. A célula 5 grava um zip parcial ao fim de cada rodada.

> Nenhum serviço de LLM em nuvem é usado: o Ollama roda dentro desta máquina virtual.

In [ ]:
# 1) Configuração — NÃO altere depois de começar
MODELOS = ['gemma4:e2b-it-qat', 'qwen3:4b-instruct-2507-q4_K_M']
HOJE_LOTE = '2026-09-24'
HOJE_310 = '2026-09-14'
DATASET_SHA256 = '07bf424a40d68392cad4a190c334b3b6144676d2927a75ae6b0b5328abf0ba42'    # gravado por scripts/empacotar_colab.py
LOTE_SHA256 = 'ac9ae34e4594e561f6a8f4c77820a6b545b91e31f6fe5abc2fdb49d8ec1d0d18'       # gravado por scripts/empacotar_colab.py
LOTE2_SHA256 = 'ba5130cbbf898dc6f8bbe8bc17d8d03cec29b032df39b88979b76d3b34f69b92'      # gravado por scripts/empacotar_colab.py
V3_SHA256 = {
 "tool_calling_v3d": "9723e508ea82ce7edd60d4853cec4a2c701835a8bf2c91fcdcc90cb433c98b39",
 "tool_calling_v3a": "bb38c7ecd8a3e67a71732bef5a6d37a5644a374d4b45d3fc19001f1fc597184d",
 "tool_calling_v3": "c6bfce53a1f6d566d22a24ea84fc786984ecb9b67ea48619661622743ffaea37",
 "saida_estruturada_v3": "fb71acc6fdde7dbf3db18466b9ec3ac2caabd7f6a84050f752d24fe4530e8b5e"
}   # pré-registro da v3 (scripts/v3/gerar_notebook_lote2.py)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2) Instala o Ollama (precisa de zstd) e sobe o servidor
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y zstd pciutils > /dev/null 2>&1 && echo "zstd instalado"
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3

import os, shutil, subprocess, time, json, urllib.request
assert shutil.which('ollama'), 'o Ollama NÃO foi instalado — veja as mensagens acima'
servidor = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT,
                            env=dict(os.environ, OLLAMA_KEEP_ALIVE='30m'))
for _ in range(40):
    try:
        v = json.load(urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=3))['version']
        print('Ollama', v, 'no ar')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('o servidor do Ollama não subiu — veja /content/ollama.log')

In [ ]:
# 3) Envia o pacote, confere datasets e a v3 congelada, instala
import os, hashlib, shutil, datetime
from google.colab import files

def sha(caminho):
    p = '/content/pfc_busca/' + caminho
    return hashlib.sha256(open(p, 'rb').read()).hexdigest() if os.path.exists(p) else None

def pacote_atual():
    return (sha('data/dataset.json') == DATASET_SHA256 and sha('data/lote_validacao.json') == LOTE_SHA256
            and sha('data/lote_validacao_2.json') == LOTE2_SHA256)

if os.path.exists('/content/pfc_busca') and not pacote_atual():
    antigo = '/content/pfc_busca_anterior_' + datetime.datetime.now().strftime('%H%M%S')
    shutil.move('/content/pfc_busca', antigo)
    print('pacote anterior movido para', antigo)
if not os.path.exists('/content/pfc_busca/pyproject.toml'):
    enviado = files.upload()
    nome = next(iter(enviado))
    !unzip -q -o "$nome" -d /content
assert pacote_atual(), 'O zip enviado NÃO é o pacote atual (hashes dos datasets). Baixe de novo dist/pfc_busca_colab.zip.'
%cd /content/pfc_busca
!pip install -q -e . 2>&1 | tail -2
import subprocess, json as _json
atual = _json.loads(subprocess.run(['python', '-c', 'import hashlib, json; from pfc_busca import v3; print(json.dumps({a: hashlib.sha256(v3.texto_hash(a).encode()).hexdigest() for a in v3.ABORDAGENS_V3}))'],
                                   capture_output=True, text=True).stdout)
assert atual == V3_SHA256, 'A v3 do pacote NÃO é a pré-registrada. Não rode: o teste perderia o valor.'
print('datasets e v3 conferidos (pré-registro ok)')
!python -m pytest -q tests/test_v3.py 2>&1 | tail -2

In [ ]:
# 4) Baixa os modelos
import subprocess as _sp
for _m in MODELOS:
    print(_m, _sp.run(['ollama', 'pull', _m], capture_output=True, text=True).stdout.strip().splitlines()[-1:])
!ollama list

In [ ]:
# 5) Rodadas (uma repetição cada)
import datetime, re, subprocess
from google.colab import files

CONJUNTOS = {'lote2': ('data/lote_validacao_2.json', 'results/lote2', HOJE_LOTE),
             'base': (None, 'results/v3_310', HOJE_310)}

def rodar(abordagem, conjunto, modelo):
    dataset, saida, hoje = CONJUNTOS[conjunto]
    cmd = ['pfc-avaliar', '--modelo', modelo, '--abordagem', abordagem, '--repeticoes', '1', '--hoje', hoje,
           '--sem-sql', '--saida', saida] + (['--dataset', dataset] if dataset else [])
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, COLUMNS='110'))
    n = 0
    for linha in proc.stdout:
        linha = linha.rstrip()
        if re.match(r'^[✓·×] r\d', linha):
            n += 1
            if linha.startswith('×') or n % 100 == 0:
                print(f'   [{n}] {linha}', flush=True)
        elif linha.strip() and not linha.startswith('avaliando'):
            print(linha, flush=True)
    return proc.wait()

def baixar_parcial():
    !cd /content && rm -f lote2_v3_modelos_colab.zip && zip -q -r lote2_v3_modelos_colab.zip pfc_busca/results/lote2 pfc_busca/results/v3_310 2>/dev/null; ls -la lote2_v3_modelos_colab.zip

rodadas = [
    ('tool_calling_v3', 'lote2', 'gemma4:e2b-it-qat'),
    ('saida_estruturada_v3', 'lote2', 'gemma4:e2b-it-qat'),
    ('tool_calling', 'lote2', 'gemma4:e2b-it-qat'),
    ('saida_estruturada', 'lote2', 'gemma4:e2b-it-qat'),
    ('tool_calling_v3', 'lote2', 'qwen3:4b-instruct-2507-q4_K_M'),
    ('saida_estruturada_v3', 'lote2', 'qwen3:4b-instruct-2507-q4_K_M'),
    ('tool_calling', 'lote2', 'qwen3:4b-instruct-2507-q4_K_M'),
    ('saida_estruturada', 'lote2', 'qwen3:4b-instruct-2507-q4_K_M'),
]
for abordagem, conjunto, modelo in rodadas:
    print(f'\n== {modelo} · {abordagem} · {conjunto} == início {datetime.datetime.now():%H:%M}', flush=True)
    codigo = rodar(abordagem, conjunto, modelo)
    print(f'   saída {codigo} às {datetime.datetime.now():%H:%M}', flush=True)
    baixar_parcial()

In [ ]:
# 6) Confere e baixa o resultado final
!python -m pfc_busca.evaluation.conferencia results/lote2/* results/v3_310/* 2>&1 | grep -v consolidado
baixar_parcial()
files.download('/content/lote2_v3_modelos_colab.zip')

## De volta ao PC

Descompacte `lote2_v3_modelos_colab.zip` na pasta que contém `pfc_busca/` (as rodadas vão para `pfc_busca/results/lote2/` e `pfc_busca/results/v3_310/`) e gere as tabelas com `python -m pfc_busca.evaluation.lote --paper ../paper_revisado`.